In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pymcts.core.players import RandomPlayer, MCTSPlayer
from pymcts.games.bridgit.neural_net import BridgitNet
from pymcts.core.self_play import batched_self_play
from pymcts.games.bridgit.config import BoardConfig, NeuralNetConfig
from pymcts.core.config import MCTSConfig
from pymcts.games.bridgit.visualizer import Visualizer
from pymcts.games.bridgit.game import BridgitGame

In [3]:
board_config = BoardConfig(size=6)
# net = BridgitNet(board_config=board_config, net_config=NeuralNetConfig(num_channels=32, num_res_blocks=4))

# trained_net_1 = BridgitNet(board_config, NeuralNetConfig(num_channels=32, num_res_blocks=4))
# checkpoint_path = "trainings/run_2026-03-28_223917/iteration_117/post_training.pt"
# trained_net_1.load_checkpoint(checkpoint_path)

# trained_net_2 = BridgitNet(board_config, NeuralNetConfig(num_channels=32, num_res_blocks=4))
# checkpoint_path = "trainings/run_2026-03-28_223917/iteration_117/post_training.pt"
# trained_net_2.load_checkpoint(checkpoint_path)

# # player_1 = MCTSPlayer(net=net, mcts_config=MCTSConfig(num_simulations=200), name="MCTS_untrained", temperature=0)
# # player_2 = MCTSPlayer(net=net, mcts_config=MCTSConfig(num_simulations=200), name="MCTS_untrained", temperature=0)
# player_1 = MCTSPlayer(net=trained_net_1, mcts_config=MCTSConfig(num_simulations=200, num_parallel_leaves=40), name="MCTS_trained_1", temperature=0)
# player_2 = MCTSPlayer(net=trained_net_2, mcts_config=MCTSConfig(num_simulations=200, num_parallel_leaves=40), name="MCTS_trained_2", temperature=0)


player_1 = MCTSPlayer.from_training_iteration("trainings/run_2026-03-29_100200/iteration_087")
player_2 = MCTSPlayer.from_training_iteration("trainings/run_2026-03-29_100200/iteration_087")

In [ ]:
from pymcts.arena import batched_arena

results = batched_arena(
    player_a=player_1, player_b=player_2,
    game_factory=lambda: BridgitGame(board_config),
    num_games=100, batch_size=10,
    swap_players=True,
)

In [6]:
Visualizer.visualize_game(results.game_records[2])

## Profiling

Profile a batched self-play run to see where time is spent (neural net forward pass, tree operations, encoding, etc.).

In [10]:
import torch
from pymcts.core.mcts import MCTS
net_config = NeuralNetConfig(num_channels=32, num_res_blocks=4, device="mps")
net = BridgitNet(board_config=board_config, net_config=net_config)
mcts = MCTS(net, MCTSConfig(num_simulations=20000, num_parallel_leaves=1000))
games = [BridgitGame(board_config) for _ in range(50)]

# Warm up (first call has overhead from MPS/CUDA compilation)
_ = mcts.search_batch(games[:2])

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True,
) as prof:
    roots = mcts.search_batch(games)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=30))

--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                        aten::to         0.20%      33.761ms        95.58%       16.012s      16.009us       1000134  
                  aten::_to_copy         0.00%     461.091us        95.38%       15.978s     253.617ms            63  
                     aten::copy_        95.37%       15.977s        95.38%       15.977s     253.603ms            63  
                    aten::detach         1.41%     235.837ms         2.48%     415.061ms       0.415us       1000071  
                          detach         1.07%     179.224ms         1.07%     179.224ms       0.179us       1000071  
                aten::lift_fresh         0.49%  

In [13]:
import cProfile
import pstats
from io import StringIO

net = BridgitNet(board_config=board_config, net_config=NeuralNetConfig(device="mps"))
mcts = MCTS(net, MCTSConfig(num_simulations=200, num_parallel_leaves=8))
games = [BridgitGame(board_config) for _ in range(16)]

# Warm up
_ = mcts.search_batch(games[:2])

with cProfile.Profile() as pr:
    roots = mcts.search_batch(games)

stats = pstats.Stats(pr, stream=(buf := StringIO()))
stats.sort_stats("cumulative").print_stats(30)
print(buf.getvalue())

         395867 function calls (392565 primitive calls) in 0.334 seconds

   Ordered by: cumulative time
   List reduced from 140 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.006    0.006    0.334    0.334 /Users/tomaszjuszczyszyn/Projects/pwr/pymcts/src/pymcts/core/mcts.py:223(search_batch)
       26    0.001    0.000    0.226    0.009 /Users/tomaszjuszczyszyn/Projects/pwr/pymcts/src/pymcts/core/mcts.py:126(_predict_batch)
       26    0.000    0.000    0.223    0.009 /Users/tomaszjuszczyszyn/Projects/pwr/pymcts/src/pymcts/core/base_neural_net.py:87(predict_batch)
       52    0.136    0.003    0.136    0.003 {method 'cpu' of 'torch._C.TensorBase' objects}
     3216    0.015    0.000    0.049    0.000 /Users/tomaszjuszczyszyn/Projects/pwr/pymcts/src/pymcts/core/mcts.py:133(_set_priors)
     3200    0.001    0.000    0.043    0.000 /Users/tomaszjuszczyszyn/Projects/pwr/pymcts/src/pymcts/core/mcts.py:161(_select_le

In [14]:
import time
from pymcts.core.self_play import batched_self_play
from pymcts.core.data import examples_from_records
from pymcts.core.players import GreedyMCTSPlayer

board_config = BoardConfig(size=5)
net_config = NeuralNetConfig(num_channels=32, num_res_blocks=4)
net = BridgitNet(board_config=board_config, net_config=net_config)
mcts_config = MCTSConfig(num_simulations=50, num_parallel_leaves=10)

# --- Phase 1: Self-play ---
t0 = time.time()
records = batched_self_play(
    net=net,
    game_factory=lambda: BridgitGame(board_config),
    mcts_config=mcts_config,
    num_games=1000,
    batch_size=100,
    temperature=1.0,
    verbose=True,
)
t_selfplay = time.time() - t0

# --- Phase 2: Training ---
examples = examples_from_records(records, lambda cfg: BridgitGame(BoardConfig(**cfg)))
t0 = time.time()
metrics = net.train_on_examples(
    examples,
    num_epochs=20,
    batch_size=64,
    verbose=True,
)
t_train = time.time() - t0

# --- Phase 3: Arena ---
player_a = GreedyMCTSPlayer(net, mcts_config, name="new")
player_b = GreedyMCTSPlayer(net, mcts_config, name="prev")
arena = Arena(player_a, player_b, game_factory=lambda: BridgitGame(board_config))
t0 = time.time()
eval_records = arena.play_games(num_games=100, verbose=True, swap_players=True)
t_arena = time.time() - t0

# --- Summary ---
total = t_selfplay + t_train + t_arena
print(f"\n{'='*40}")
print(f"Self-play:  {t_selfplay:6.1f}s  ({100*t_selfplay/total:.0f}%)")
print(f"Training:   {t_train:6.1f}s  ({100*t_train/total:.0f}%)")
print(f"Arena:      {t_arena:6.1f}s  ({100*t_arena/total:.0f}%)")
print(f"Total:      {total:6.1f}s")


Self-play:   0%|          | 0/1000 [00:00<?, ?it/s]

Training:   0%|          | 0/20 [00:00<?, ?it/s]

new vs prev:   0%|          | 0/100 [00:00<?, ?it/s]


Self-play:   196.6s  (28%)
Training:    480.6s  (69%)
Arena:        21.5s  (3%)
Total:       698.6s
